In [1]:
#cell1
import sqlite3

conn = sqlite3.connect("campus.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS events (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp     TEXT NOT NULL,
    module        TEXT NOT NULL,
    camera        TEXT NOT NULL,
    event_type    TEXT NOT NULL,
    severity      TEXT NOT NULL,
    plate         TEXT DEFAULT '',
    evidence_url  TEXT DEFAULT '',
    status        TEXT DEFAULT 'New'
)""")

cur.execute("""
CREATE TABLE IF NOT EXISTS cameras (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    name        TEXT UNIQUE NOT NULL,
    status      TEXT DEFAULT 'Online',
    stream_url  TEXT DEFAULT ''
)""")

conn.commit()
print("Tables ban gaye")

Tables ban gaye


In [2]:
#cell2 cameras aur kuch test events daalo
from datetime import datetime

cameras = ["Main Gate", "Parking A", "Library Road", "Hostel Gate", "Lab Block", "Back Gate"]
for c in cameras:
    cur.execute("INSERT OR IGNORE INTO cameras (name) VALUES (?)", (c,))

def add_event(module, camera, event_type, severity, plate="", evidence_url=""):
    cur.execute(
        "INSERT INTO events (timestamp, module, camera, event_type, severity, plate, evidence_url) "
        "VALUES (?,?,?,?,?,?,?)",
        (datetime.now().isoformat(timespec="seconds"), module, camera, event_type, severity, plate, evidence_url))
    conn.commit()

add_event("Vehicle + ANPR", "Main Gate", "Authorized vehicle", "Low", plate="UK07A1234")
add_event("Helmet + Triple Riding", "Parking A", "No helmet", "Medium")
add_event("Threat Detection", "Lab Block", "Knife detected", "High")
add_event("Intrusion", "Back Gate", "Restricted zone entry", "High")
print("Events add ho gaye")

Events add ho gaye


In [3]:
#cell3
import pandas as pd
pd.read_sql("SELECT * FROM events ORDER BY id DESC", conn)

,id,timestamp,module,camera,event_type,severity,plate,evidence_url,status
0,4,2026-10-16T15:46:26,Intrusion,Back Gate,Restricted zone entry,High,,,New
1,3,2026-10-16T15:46:26,Threat Detection,Lab Block,Knife detected,High,,,New
2,2,2026-10-16T15:46:26,Helmet + Triple Riding,Parking A,No helmet,Medium,,,New
3,1,2026-10-16T15:46:26,Vehicle + ANPR,Main Gate,Authorized vehicle,Low,UK07A1234,,New


In [4]:
#cell4
# Sirf High severity
print(pd.read_sql("SELECT * FROM events WHERE severity='High'", conn))

# Module ke hisaab se count
print(pd.read_sql("SELECT module, COUNT(*) AS total FROM events GROUP BY module", conn))

# Alert ko Acknowledged mark karna
cur.execute("UPDATE events SET status='Acknowledged' WHERE id=1")
conn.commit()

   id            timestamp            module     camera  \
0   3  2026-10-16T15:46:26  Threat Detection  Lab Block   
1   4  2026-10-16T15:46:26         Intrusion  Back Gate   

              event_type severity plate evidence_url status  
0         Knife detected     High                       New  
1  Restricted zone entry     High                       New  
                   module  total
0  Helmet + Triple Riding      1
1               Intrusion      1
2        Threat Detection      1
3          Vehicle + ANPR      1


In [5]:
#cell5
from google.colab import files
files.download("campus.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>